In [1]:
import Setup.set_sy as set_sy
import requests
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
# -------------------------
# CONFIG
# -------------------------
SHOP = "wooden-ships"
SPREADSHEET_ID = "1CX6tjxos0N2p_YRmrgo6sA7KSPM5bZnBdyaQZuJWoCk"
RANGE = "Sheet1!A1"



# -------------------------
# GOOGLE SHEETS SETUP
# -------------------------
creds = Credentials.from_service_account_file(
    "credentials/dialy-report-automation-e20c53e67542.json",
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)


service = build("sheets", "v4", credentials=creds)
sheet = service.spreadsheets()


# -------------------------
# SHOPIFY GRAPHQL SETUP
# -------------------------
url = f"https://{SHOP}.myshopify.com/admin/api/2024-01/graphql.json"

headers = {
    "X-Shopify-Access-Token": set_sy.get_token(),
    "Content-Type": "application/json"
}


query = """
query ($cursor: String) {
  products(first: 250, after: $cursor, query: "status:active OR status:draft") {
    edges {
      node {
        id
        title
        status
        variants(first: 1) {
          edges {
            node {
              selectedOptions {
                name
                value
              }
            }
          }
        }
      }
    }
    pageInfo {
      hasNextPage
      endCursor
    }
  }
}
"""

/Users/woodenship/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/woodenship/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/woodenship/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python versi

In [2]:
rows = []

# Header row (matches your screenshot)
rows.append([
    "Style",
    "Color",
    "FP/DC",
    "Product ID",
    "Page Status",
    "Current Production Type"
])

cursor = None

while True:
    response = requests.post(
        url,
        headers=headers,
        json={"query": query, "variables": {"cursor": cursor}}
    )

    data = response.json()["data"]["products"]

    for edge in data["edges"]:
        p = edge["node"]

        product_id = p["id"].split("/")[-1]
        title = p["title"]
        status = p["status"]

        # Extract first variant color
        color = ""
        variants = p["variants"]["edges"]

        if variants:
            for opt in variants[0]["node"]["selectedOptions"]:
                if opt["name"].lower() in ["color", "colour"]:
                    color = opt["value"]
                    break

        rows.append([
            title,
            color,
            "",             # FP/DC
            product_id,
            status,
            ""              # Current Production Type
        ])

    if not data["pageInfo"]["hasNextPage"]:
        break

    cursor = data["pageInfo"]["endCursor"]

print(f"Fetched {len(rows)-1} products")




Fetched 14646 products


In [ ]:
import gspread 
import all_function_list_underdev as afl


afl._get_sheet_values(
    "1CX6tjxos0N2p_YRmrgo6sA7KSPM5bZnBdyaQZuJWoCk",
    worksheet_index="Color list",
    range_name="A:B",
    use_all_values=True

)